# Plug in LM Sample corners, as well as the partice_sample positions computed from the compute_global_coordinates code from SAM2 output- transforms SEM coordinates into LM coordinates

In [12]:
import pandas as pd
import numpy as np

# ─── Helper functions ────────────────────────────────────────────────────────
def load_sample_corners(corners_csv):
    """
    Read just the top‐left & top‐right sample corners from CSV,
    returning a dict with their stage coordinates.
    """
    df = pd.read_csv(corners_csv)
    corners = {
        row.Name.strip(): np.array([row.X, row.Y])
        for row in df.itertuples(index=False)
    }
    required = ["Sample Top Left", "Sample Top Right"]
    for name in required:
        if name not in corners:
            raise KeyError(f"Missing corner '{name}' in {corners_csv}")
    return corners

def compute_sample_transform(corners):
    """
    Builds a rotation matrix R and origin so that
       sample_coords = R @ (stage_coords - origin)
    where origin = top‐left corner, and the x‐axis of the sample
    is aligned along the vector from top‐left → top‐right.
    """
    tl = corners["Sample Top Left"]
    tr = corners["Sample Top Right"]
    v = tr - tl
    angle = np.arctan2(v[1], v[0])      # angle of top edge
    cos_a, sin_a = np.cos(-angle), np.sin(-angle)
    R = np.array([[ cos_a, -sin_a],
                  [ sin_a,  cos_a]])
    return R, tl

# ─── Load your LM‐measured sample corners & build transform ────────────────
lm_csv     = "LM_sample_stage_corners.csv"
corners_lm = load_sample_corners(lm_csv)
R_lm, origin_lm = compute_sample_transform(corners_lm)

# ─── Read the SEM‐derived sample‐coords CSV ──────────────────────────────────
particles = pd.read_csv("particle_sample_positions_October8_plain.csv")

# ─── Map sample coords back into LM‐stage coords: inv(R) * sample + origin ──
S = particles[["sample_x_m", "sample_y_m"]].to_numpy()   # (N×2)
stage_pts = (R_lm.T @ S.T).T + origin_lm[np.newaxis, :]

particles["lm_stage_x_m"] = stage_pts[:, 0]
particles["lm_stage_y_m"] = stage_pts[:, 1]

# ─── Save the corrected CSV ─────────────────────────────────────────────────
out_csv = "particle_LM_stage_positions_plain_October8.csv"
particles.to_csv(out_csv, index=False)
print(f"✅ Saved corrected LM‐stage positions to {out_csv}")

# Quick check
print(particles[["lm_stage_x_m", "lm_stage_y_m"]].head())


✅ Saved corrected LM‐stage positions to particle_LM_stage_positions_plain_October8.csv
   lm_stage_x_m  lm_stage_y_m
0      0.002353     -0.001010
1      0.002359     -0.001017
2      0.002357     -0.001017
3      0.002341     -0.001033
4      0.002356     -0.001032
